In [ ]:
import os
from google.colab import userdata

os.environ.setdefault('WCD_PLANNER_PROVIDER','ollama')
os.environ['WCD_OLLAMA_API_KEY'] = userdata.get('OLLAMA_API_KEY')
if not os.environ.get('WCD_OLLAMA_API_KEY'):
    raise RuntimeError('Missing WCD_OLLAMA_API_KEY. Configure it before running agentic discovery.')


# Webcam Discovery Feed Discovery Colab Test

In [ ]:
!python --version
!pip install -e .
!webcam-discovery run-agentic --help

In [ ]:
import requests, json
rows=[]
for d in range(1,13):
    u=f'https://cwwp2.dot.ca.gov/data/d{d}/cctv/cctvStatusD{d:02d}.json'
    try:
        j=requests.get(u,timeout=20).json()
        rows.extend(j if isinstance(j,list) else j.get('data',[]))
    except Exception:
        pass
print('records',len(rows))

In [ ]:
!webcam-discovery run-agentic "Get me public live traffic cameras from California" --enable-feed-discovery --max-feed-endpoints 100 --max-feed-records 3000 --max-stream-candidates 2500 --per-source-stream-cap 500 --preserve-direct-streams --max-streams 500 --enable-visual-analysis

In [ ]:
import json, os, zipfile
for p in ['camera.geojson','cameras.md','logs/discovery_funnel.json','logs/source_performance.json','map.html']:
    print(p, os.path.exists(p))
if os.path.exists('logs/discovery_funnel.json'):
    print(json.load(open('logs/discovery_funnel.json')))
with zipfile.ZipFile('webcam_artifacts.zip','w') as z:
    for root,_,files in os.walk('logs'):
        for f in files: z.write(os.path.join(root,f))
    for root,_,files in os.walk('candidates'):
        for f in files: z.write(os.path.join(root,f))
    for p in ['camera.geojson','cameras.md','map.html']:
        if os.path.exists(p): z.write(p)
print('wrote webcam_artifacts.zip')